# 03 · Batching with CUDA-Q

Many workloads, such as VQE energies, parameter sweeps and large Hamiltonians, consist of **lots of independent small jobs**. CUDA-Q can batch them and spread them across GPUs:

| Part | What is batched | How |
|---|---|---|
| 1 | One observable $\langle Z^{\otimes n}\rangle$ | `cudaq.observe` |
| 2 | A Hamiltonian with ~20 000 terms | `observe(..., execution=cudaq.parallel.thread)` on `mqpu` |
| 3 | The same Hamiltonian on tensor-network backends | `tensornet`, `tensornet-mps` |
| 4 | 10 000 parameter sets | array broadcasting + `observe_async` |

**Kernel:** select **Python (CUDA-Q GPU)**. To see the multi-QPU speedups, request **2 or more GPUs** for the session (e.g. `--gres=gpu:h100-80:2`); with 1 GPU everything still runs on a single QPU.

## Setup: GHZ kernel (no measurement) and Pauli-string observables

In [ ]:
import cudaq
from cudaq import spin
import numpy as np
import timeit

print("CUDA-Q version:", cudaq.__version__)
print("GPUs visible  :", cudaq.num_available_gpus())

QUBITS = 30


# GHZ state preparation, without measurement, for use with cudaq.observe
@cudaq.kernel
def ghz_kernel(qubit_count: int):
    qubits = cudaq.qvector(qubit_count)
    h(qubits[0])
    for i in range(1, qubit_count):
        x.ctrl(qubits[0], qubits[i])


def z_power_n(n: int):
    """Z(0) * Z(1) * ... * Z(n-1)"""
    if n <= 0:
        raise ValueError("n must be >= 1")
    op = spin.z(0)
    for i in range(1, n):
        op = op * spin.z(i)
    return op


def x_power_n(n: int):
    """X(0) * X(1) * ... * X(n-1)"""
    if n <= 0:
        raise ValueError("n must be >= 1")
    op = spin.x(0)
    for i in range(1, n):
        op = op * spin.x(i)
    return op


def y_power_n(n: int):
    """Y(0) * Y(1) * ... * Y(n-1)"""
    if n <= 0:
        raise ValueError("n must be >= 1")
    op = spin.y(0)
    for i in range(1, n):
        op = op * spin.y(i)
    return op


def timed_observe(kernel, op, *args, **kwargs):
    start = timeit.default_timer()
    value = cudaq.observe(kernel, op, *args, **kwargs).expectation()
    return value, timeit.default_timer() - start

## Part 1 · A single observable: $\langle Z^{\otimes n}\rangle$
For a GHZ state every qubit is either all 0 or all 1, so the product $Z_0 Z_1 \cdots Z_{n-1}$ always equals $+1$ for even parity. **The expectation value should be close to 1.**

In [ ]:
cudaq.set_target("nvidia")

value, sec = timed_observe(ghz_kernel, z_power_n(QUBITS), QUBITS)
print(f"<Z^{QUBITS}> = {value:.6f}   time: {sec:.2f} s")

With a single term there's nothing to split, so `mqpu` gives the same answer in about the same time:

In [ ]:
cudaq.set_target("nvidia", option="mqpu")
print("Number of QPUs:", cudaq.get_target().num_qpus())

value, sec = timed_observe(ghz_kernel, z_power_n(QUBITS), QUBITS,
                           execution=cudaq.parallel.thread)
print(f"<Z^{QUBITS}> = {value:.6f}   time: {sec:.2f} s")

## Part 2 · Batching a large Hamiltonian over QPUs
Now build a big Hamiltonian:

$$H = \underbrace{\sum_{k=1}^{10\,000} c_k P_k}_{\text{random Pauli strings}} + Z^{\otimes n} + X^{\otimes n} + Y^{\otimes n}$$

`cudaq.observe` measures each term separately, so the cost grows with the number of terms. Passing `execution=cudaq.parallel.thread` on the `mqpu` target **splits the terms into batches, one per GPU**, and sums the results.

In [ ]:
h_op = cudaq.SpinOperator.random(QUBITS, 10000, seed=13)
h_op += z_power_n(QUBITS)
h_op += x_power_n(QUBITS)
h_op += y_power_n(QUBITS)

print(f"Total number of terms in H: {h_op.term_count}")

### 1 QPU (single GPU)
> ⏱ Reference: **68.5 s** on one V100 (`GHZ_batch_H/1QPU`).

In [ ]:
cudaq.set_target("nvidia")

value_1qpu, sec_1qpu = timed_observe(ghz_kernel, h_op, QUBITS)
print(f"<H> = {value_1qpu:.10f}   time: {sec_1qpu:.2f} s")

### All QPUs (`mqpu` + `cudaq.parallel.thread`)
> ⏱ Reference: **35.2 s** on 2 × V100 (`GHZ_batch_H/2QPU`), almost a 2× speedup. The `gpu_log_*` files show both GPUs at ~100 % utilisation during the 2-QPU run versus only GPU 0 for the 1-QPU run.

In [ ]:
cudaq.set_target("nvidia", option="mqpu")
num_qpus = cudaq.get_target().num_qpus()
print("Number of QPUs:", num_qpus)

value_mqpu, sec_mqpu = timed_observe(ghz_kernel, h_op, QUBITS,
                                     execution=cudaq.parallel.thread)
print(f"<H> = {value_mqpu:.10f}   time: {sec_mqpu:.2f} s")
print(f"speedup vs 1 QPU: {sec_1qpu / sec_mqpu:.2f}x on {num_qpus} QPU(s)")

You can watch GPU utilisation while a cell runs by opening a **Terminal** and running:

```bash
nvidia-smi --query-gpu=index,name,utilization.gpu,utilization.memory,memory.used,memory.free --format=csv -l 5
```

For multi-node runs, use the MPI variant: `execution=cudaq.parallel.mpi` in a script launched with `mpirun` (after `cudaq.mpi.initialize()`, and `cudaq.mpi.finalize()` at the end).

## Part 3 · The same Hamiltonian on tensor-network backends
Tensor-network backends also evaluate each term, so a 20 000-term Hamiltonian is slow here. These cells are **optional**.

> ⏱ Reference (1 × V100): `tensornet` **403 s**, `tensornet-mps` **262 s** (`GHZ_batch_H/tensornet-1QPU`, `tensornet-mps-1QPU`).

In [ ]:
cudaq.set_target("tensornet")

value_tn, sec_tn = timed_observe(ghz_kernel, h_op, QUBITS)
print(f"tensornet     <H> = {value_tn:.10f}   time: {sec_tn:.2f} s")

In [ ]:
cudaq.set_target("tensornet-mps")

value_mps, sec_mps = timed_observe(ghz_kernel, h_op, QUBITS)
print(f"tensornet-mps <H> = {value_mps:.10f}   time: {sec_mps:.2f} s")

## Part 4 · Batching parameter sets
The other common batch is **one circuit, many parameters**, as in a VQE scan or training loop. Here we evaluate $\langle Z_0\rangle$ for 10 000 parameter sets of a 5-qubit circuit.

### Broadcasting on one GPU
Passing a 2-D NumPy array as the argument makes `cudaq.observe` run the kernel once per row and return a list of results.

In [ ]:
cudaq.set_target("nvidia")

qubit_count = 5
sample_count = 10000
h_obs = spin.z(0)

parameters = np.random.default_rng(13).uniform(low=0, high=1,
                                               size=(sample_count, qubit_count))


@cudaq.kernel
def rx_kernel(params: list[float]):
    qubits = cudaq.qvector(5)
    for i in range(5):
        rx(params[i], qubits[i])


start = timeit.default_timer()
results = cudaq.observe(rx_kernel, h_obs, parameters)
sec_bcast = timeit.default_timer() - start
print(f"{len(results)} expectation values, first 3: "
      f"{[round(r.expectation(), 4) for r in results[:3]]}")
print(f"Single GPU, broadcast: {sec_bcast:.3f} s")   # reference (V100): 3.75 s

### Asynchronous batches across QPUs
Split the parameter sets into 4 batches and send each batch to a QPU with `observe_async`. The calls return immediately with a future, and `.get()` waits for the result.

In [ ]:
cudaq.set_target("nvidia", option="mqpu")
num_gpus = cudaq.num_available_gpus()

batches = np.split(parameters, 4)
print('Split parameters into', len(batches), 'batches of',
      [b.shape[0] for b in batches])

start = timeit.default_timer()
futures = []
for i, batch in enumerate(batches):
    qpu_id = i * num_gpus // len(batches)
    for params in batch:
        futures.append(cudaq.observe_async(rx_kernel, h_obs, params, qpu_id=qpu_id))
values = [f.get().expectation() for f in futures]
sec_async = timeit.default_timer() - start
print(f"{len(values)} expectation values on {num_gpus} QPU(s): {sec_async:.3f} s")   # reference (2 x V100): 1.60 s

## Summary

| Batch type | API | Scales with |
|---|---|---|
| Many Hamiltonian terms | `cudaq.observe(..., execution=cudaq.parallel.thread)` on `nvidia` `mqpu` | number of GPUs |
| Many parameter sets, 1 GPU | `cudaq.observe(kernel, H, params_2d_array)` | GPU throughput |
| Many parameter sets, many GPUs | `cudaq.observe_async(..., qpu_id=i)` / `cudaq.sample_async` | number of GPUs |
| Across nodes | `execution=cudaq.parallel.mpi` + `mpirun` | number of GPUs × nodes |

### Reference timings (V100, from `GHZ_batch_H`)

| Run | Time (s) |
|---|---|
| 20 006-term H, `nvidia`, 1 QPU | 68.5 |
| 20 006-term H, `nvidia` `mqpu`, 2 QPUs | 35.2 |
| 20 006-term H, `tensornet` | 403.1 |
| 20 006-term H, `tensornet-mps` | 261.6 |